# Forklift Monocular Depth Estimation POC

`data/movie_preprocess` 配下の前処理済み動画から等間隔にフレームを抜き出し、単眼深度推定でフォークリフト車体領域を推定します。

`VIDEO_TARGETS` に複数の動画を指定すると、動画ごとにサンプリング、深度推定、共通手前領域の抽出、ログ出力を一括で行います。


## 処理の流れ

1. `data/movie_preprocess` 配下の `*_front.mp4` / `*_rear.mp4` を一覧化します。
2. `VIDEO_TARGETS` で指定した動画から、動画全体にわたって等間隔にフレームを抽出します。
3. Hugging Face Transformers の `depth-estimation` pipeline で各フレームの相対深度を推定します。
4. 各フレームで手前に出た領域を求め、全フレームで共通して手前に出た領域をフォークリフト車体領域として抽出します。
5. 車体領域の面積、bbox、サンプリング時刻をログと表で出力します。


In [ ]:
# ------------------------------------------------------------
# セル概要: パッケージ確認
# ------------------------------------------------------------
# - transformers / torch / OpenCV / matplotlib など、単眼深度推定に必要なライブラリを確認します。
# - 未導入の環境では、下記を有効化してから実行してください。
# ------------------------------------------------------------

%pip install transformers torch pillow opencv-python matplotlib pandas numpy


In [ ]:
# ------------------------------------------------------------
# セル概要: ライブラリ読み込み
# ------------------------------------------------------------
# - Path / OpenCV / numpy / pandas / matplotlib などを読み込みます。
# - transformers と torch はモデル初期化セルで遅延importします。
# ------------------------------------------------------------

from __future__ import annotations

from pathlib import Path
from typing import Any

import sys

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image


def _find_import_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "forklift_features").exists():
            return candidate
    raise FileNotFoundError(f"Could not find src/forklift_features from {start}.")


_IMPORT_ROOT = _find_import_root()
src_path = str(_IMPORT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from forklift_features.paths import find_project_root, safe_path_part


In [ ]:
# ------------------------------------------------------------
# セル概要: 対象動画と推定設定
# ------------------------------------------------------------
# - VIDEO_TARGETS に、処理したい動画をまとめて指定します。
# - video は data/movie_preprocess からの相対パス、ファイル名、または sample_id として指定できます。
# - sample_id 指定では category / environment は不要です。view は front/rear の選択に使います。
# ------------------------------------------------------------

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
MOVIE_PREPROCESS_DIR = DATA_DIR / "movie_preprocess"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "monocular_depth_estimation"

# 初回実行時はHugging Face Hubからモデルをダウンロードします。
DEPTH_MODEL_NAME = "depth-anything/Depth-Anything-V2-Small-hf"

# 深度ヒートマップの見え方を調整します。モデル出力は相対深度で、メートル単位ではありません。
DEPTH_CMAP = "magma_r"
DEPTH_PERCENTILE_RANGE = (2.0, 98.0)
OVERLAY_ALPHA = 0.45
SAVE_RESULT_IMAGES = False

# 車体領域推定用のサンプリング・マスク設定。
# DEPTH_NEAR_THRESHOLD_METHOD='multi_otsu' では、正規化深度を遠/中/近の3クラスに分け、中+近を手前扱いします。
# DEPTH_NEAR_THRESHOLD_METHOD='otsu' では、正規化深度からフレームごとに2値の手前判定閾値を自動推定します。
# DEPTH_NEAR_THRESHOLD_METHOD='manual' の場合だけ DEPTH_NEAR_THRESHOLD を使います。
FRAME_SAMPLE_COUNT = 12
FRAME_SAMPLE_START_SEC: int | float | str | None = 0.0
FRAME_SAMPLE_END_SEC: int | float | str | None = None
DEPTH_NEAR_THRESHOLD_METHOD = "multi_otsu"
DEPTH_NEAR_THRESHOLD: float | None = None
VEHICLE_COMMON_MIN_RATIO = 0.8
VEHICLE_MIN_COMPONENT_AREA_RATIO = 0.0005
VEHICLE_MORPH_OPEN_KERNEL = 5
VEHICLE_MORPH_CLOSE_KERNEL = 0
DISPLAY_PER_FRAME_DEPTH_RESULTS = False
DISPLAY_VEHICLE_REGION_RESULTS = True

# frame_sample_count / start_time / end_time は対象ごとに上書きできます。
VIDEO_TARGETS: list[dict[str, Any]] = [
    # {
    #     "video": "normal/outdoor/1001_front.mp4",
    #     "frame_sample_count": 12,
    # },
    {
        "sample_id": "001_後進時にトラックに衝突",
        "view": "front",
        "frame_sample_count": 6,
    },
    # {
    #     "sample_id": "071_旋回時、ラックに衝突",
    #     "view": "front",
    #     "frame_sample_count": 12,
    # },
    # {
    #     "sample_id": "036_後進時、壁に衝突",
    #     "view": "front",
    #     "frame_sample_count": 12,
    # }
]

print(f"PROJECT_ROOT        : {PROJECT_ROOT}")
print(f"MOVIE_PREPROCESS_DIR: {MOVIE_PREPROCESS_DIR}")
print(f"DEPTH_MODEL_NAME    : {DEPTH_MODEL_NAME}")
print(f"video targets       : {len(VIDEO_TARGETS)}")


## 1. 動画一覧


In [ ]:
# ------------------------------------------------------------
# セル概要: 前処理済み動画一覧の作成
# ------------------------------------------------------------
# - data/movie_preprocess 配下の mp4 を走査し、category / environment / sample_id / view を整理します。
# - VIDEO_TARGETS を書くときのパス確認に使います。
# ------------------------------------------------------------

def split_preprocess_video_name(path: Path) -> tuple[str | None, str | None]:
    for suffix in ("_front.mp4", "_rear.mp4"):
        if path.name.endswith(suffix):
            return path.name[: -len(suffix)], suffix.removeprefix("_").removesuffix(".mp4")
    return None, None


def build_video_inventory(movie_preprocess_dir: Path = MOVIE_PREPROCESS_DIR) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    if not movie_preprocess_dir.exists():
        raise FileNotFoundError(f"movie_preprocess directory not found: {movie_preprocess_dir}")

    for video_path in sorted(movie_preprocess_dir.glob("*/*/*.mp4")):
        sample_id, view = split_preprocess_video_name(video_path)
        if sample_id is None or view is None:
            continue
        rows.append({
            "category": video_path.parent.parent.name,
            "environment": video_path.parent.name,
            "sample_id": sample_id,
            "view": view,
            "relative_path": str(video_path.relative_to(movie_preprocess_dir)),
            "path": video_path,
        })
    return pd.DataFrame(rows)


video_inventory_df = build_video_inventory()
print(f"videos: {len(video_inventory_df)}")
display(video_inventory_df.head(20))


## 2. フレーム抽出


In [ ]:
# ------------------------------------------------------------
# セル概要: 動画解決と指定時刻フレーム抽出
# ------------------------------------------------------------
# - VIDEO_TARGETS の video または sample_id 指定を実ファイルパスへ解決します。
# - 秒・mm:ss・hh:mm:ss の指定時刻を秒へ変換し、OpenCVで該当フレームを取得します。
# ------------------------------------------------------------

def parse_time_seconds(value: int | float | str) -> float:
    if isinstance(value, (int, float)):
        seconds = float(value)
    else:
        text = str(value).strip()
        if not text:
            raise ValueError("time value is empty")
        parts = text.split(":")
        if len(parts) == 1:
            seconds = float(parts[0])
        elif len(parts) == 2:
            minutes, sec = parts
            seconds = int(minutes) * 60 + float(sec)
        elif len(parts) == 3:
            hours, minutes, sec = parts
            seconds = int(hours) * 3600 + int(minutes) * 60 + float(sec)
        else:
            raise ValueError(f"unsupported time format: {value!r}")
    if seconds < 0:
        raise ValueError(f"time must be non-negative: {value!r}")
    return seconds


def resolve_sample_video_path(
    sample_id: str,
    view: str = "front",
    movie_preprocess_dir: Path = MOVIE_PREPROCESS_DIR,
    category: str | None = None,
    environment: str | None = None,
) -> Path:
    filename = f"{sample_id}_{view}.mp4"
    if category and environment:
        candidate = movie_preprocess_dir / str(category) / str(environment) / filename
        if candidate.exists():
            return candidate.resolve()
        raise FileNotFoundError(f"video not found: {candidate}")

    matches = sorted(movie_preprocess_dir.glob(f"*/*/{filename}"))
    if len(matches) == 1:
        return matches[0].resolve()
    if not matches:
        raise FileNotFoundError(f"video not found under {movie_preprocess_dir}: {filename}")
    match_text = "\n".join(str(path.relative_to(movie_preprocess_dir)) for path in matches[:20])
    raise ValueError(f"sample_id/view is ambiguous: {sample_id}_{view}\n{match_text}")


def resolve_video_path(spec: dict[str, Any], movie_preprocess_dir: Path = MOVIE_PREPROCESS_DIR) -> Path:
    if "video" in spec:
        video = Path(str(spec["video"]))
        candidates = []
        if video.is_absolute():
            candidates.append(video)
        else:
            candidates.extend([movie_preprocess_dir / video, PROJECT_ROOT / video])
        for candidate in candidates:
            if candidate.exists():
                return candidate.resolve()

        filename_matches = sorted(movie_preprocess_dir.glob(f"*/*/{video.name}"))
        if len(filename_matches) == 1:
            return filename_matches[0].resolve()
        if len(filename_matches) > 1:
            matches = "\n".join(str(path.relative_to(movie_preprocess_dir)) for path in filename_matches[:20])
            raise ValueError(f"video name is ambiguous: {video.name}\n{matches}")
        if not video.suffix:
            return resolve_sample_video_path(str(spec["video"]), str(spec.get("view", "front")), movie_preprocess_dir)
        raise FileNotFoundError(f"video not found under {movie_preprocess_dir}: {spec['video']}")

    sample_id = spec.get("sample_id")
    if not sample_id:
        raise ValueError(f"spec must contain 'video' or 'sample_id': {spec}")
    return resolve_sample_video_path(
        str(sample_id),
        str(spec.get("view", "front")),
        movie_preprocess_dir,
        category=spec.get("category"),
        environment=spec.get("environment"),
    )


def read_frame_at_time(video_path: Path, time_value: int | float | str) -> tuple[np.ndarray, dict[str, Any]]:
    seconds = parse_time_seconds(time_value)
    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")

    try:
        fps = float(capture.get(cv2.CAP_PROP_FPS) or 0.0)
        frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
        width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
        height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)
        if fps <= 0:
            raise RuntimeError(f"Could not read FPS from video: {video_path}")

        requested_frame_index = int(round(seconds * fps))
        if frame_count > 0:
            frame_index = min(requested_frame_index, max(frame_count - 1, 0))
        else:
            frame_index = requested_frame_index

        capture.set(cv2.CAP_PROP_POS_FRAMES, frame_index)
        ok, frame_bgr = capture.read()
        if not ok or frame_bgr is None:
            raise RuntimeError(f"Could not read frame {frame_index} from {video_path}")
        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

        actual_frame_index = int(capture.get(cv2.CAP_PROP_POS_FRAMES) or (frame_index + 1)) - 1
        metadata = {
            "video_path": video_path,
            "relative_path": str(video_path.relative_to(MOVIE_PREPROCESS_DIR)) if video_path.is_relative_to(MOVIE_PREPROCESS_DIR) else str(video_path),
            "requested_time_sec": seconds,
            "requested_time": time_value,
            "fps": fps,
            "frame_count": frame_count,
            "duration_sec": frame_count / fps if frame_count > 0 else np.nan,
            "frame_index": actual_frame_index,
            "width": width,
            "height": height,
        }
        return frame_rgb, metadata
    finally:
        capture.release()


def read_video_metadata(video_path: Path) -> dict[str, Any]:
    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")
    try:
        fps = float(capture.get(cv2.CAP_PROP_FPS) or 0.0)
        frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
        width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
        height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)
        if fps <= 0:
            raise RuntimeError(f"Could not read FPS from video: {video_path}")
        return {
            "video_path": video_path,
            "relative_path": str(video_path.relative_to(MOVIE_PREPROCESS_DIR)) if video_path.is_relative_to(MOVIE_PREPROCESS_DIR) else str(video_path),
            "fps": fps,
            "frame_count": frame_count,
            "duration_sec": frame_count / fps if frame_count > 0 else np.nan,
            "width": width,
            "height": height,
        }
    finally:
        capture.release()


def sample_video_times(video_path: Path, spec: dict[str, Any]) -> tuple[list[float], dict[str, Any]]:
    metadata = read_video_metadata(video_path)
    fps = float(metadata["fps"])
    duration_sec = float(metadata["duration_sec"])
    frame_count = int(metadata["frame_count"])
    if frame_count <= 0 or not np.isfinite(duration_sec):
        raise RuntimeError(f"Could not determine duration for video: {video_path}")

    start_value = spec.get("start_time", spec.get("start_time_sec", FRAME_SAMPLE_START_SEC))
    end_value = spec.get("end_time", spec.get("end_time_sec", FRAME_SAMPLE_END_SEC))
    sample_count = int(spec.get("frame_sample_count", spec.get("sample_count", FRAME_SAMPLE_COUNT)))
    if sample_count <= 0:
        raise ValueError(f"frame_sample_count must be positive: {sample_count}")

    last_time_sec = max(0.0, duration_sec - 1.0 / max(fps, 1e-6))
    start_sec = parse_time_seconds(start_value) if start_value is not None else 0.0
    end_sec = parse_time_seconds(end_value) if end_value is not None else last_time_sec
    start_sec = min(max(0.0, start_sec), last_time_sec)
    end_sec = min(max(start_sec, end_sec), last_time_sec)

    if sample_count == 1 or end_sec <= start_sec:
        candidate_times = np.array([(start_sec + end_sec) * 0.5], dtype=float)
    else:
        candidate_times = np.linspace(start_sec, end_sec, num=sample_count, dtype=float)

    sampled_times: list[float] = []
    seen_frame_indices: set[int] = set()
    for time_sec in candidate_times:
        frame_index = min(int(round(float(time_sec) * fps)), max(frame_count - 1, 0))
        if frame_index in seen_frame_indices:
            continue
        seen_frame_indices.add(frame_index)
        sampled_times.append(float(frame_index / fps))
    return sampled_times, metadata


def iter_video_requests(video_targets: list[dict[str, Any]]) -> list[dict[str, Any]]:
    requests: list[dict[str, Any]] = []
    for spec in video_targets:
        video_path = resolve_video_path(spec)
        sampled_times, metadata = sample_video_times(video_path, spec)
        requests.append({
            "spec": spec,
            "video_path": video_path,
            "sampled_times": sampled_times,
            "metadata": metadata,
        })
    return requests


## 3. 単眼深度推定


In [ ]:
# ------------------------------------------------------------
# セル概要: 深度推定モデルと描画用変換
# ------------------------------------------------------------
# - transformers の depth-estimation pipeline を初期化します。
# - モデル出力の相対深度を元フレームサイズへ戻し、Notebook表示用のカラー画像へ変換します。
# ------------------------------------------------------------

def load_depth_estimator(model_name: str = DEPTH_MODEL_NAME):
    try:
        import torch
        from transformers import pipeline
    except ImportError as exc:
        raise ImportError(
            "Depth estimation requires transformers and torch. "
            "Run `%pip install transformers torch pillow opencv-python matplotlib pandas numpy`."
        ) from exc

    device = 0 if torch.cuda.is_available() else -1
    print(f"loading depth model: {model_name} (device={device})")
    return pipeline("depth-estimation", model=model_name, device=device)


def depth_result_to_array(depth_result: dict[str, Any], target_size: tuple[int, int]) -> np.ndarray:
    target_width, target_height = target_size
    if "predicted_depth" in depth_result:
        predicted = depth_result["predicted_depth"]
        if hasattr(predicted, "detach"):
            depth = predicted.detach().cpu().numpy()
        else:
            depth = np.asarray(predicted)
        depth = np.squeeze(depth).astype(np.float32)
    elif "depth" in depth_result:
        depth = np.asarray(depth_result["depth"], dtype=np.float32)
    else:
        raise KeyError(f"depth-estimation result keys are unsupported: {sorted(depth_result.keys())}")

    if depth.shape[:2] != (target_height, target_width):
        depth = cv2.resize(depth, (target_width, target_height), interpolation=cv2.INTER_CUBIC)
    return depth.astype(np.float32)


def normalize_depth_for_display(
    depth: np.ndarray,
    percentile_range: tuple[float, float] = DEPTH_PERCENTILE_RANGE,
) -> np.ndarray:
    finite = depth[np.isfinite(depth)]
    if finite.size == 0:
        return np.zeros(depth.shape, dtype=np.float32)

    lower, upper = np.percentile(finite, percentile_range)
    if not np.isfinite(lower) or not np.isfinite(upper) or upper <= lower:
        lower, upper = float(np.nanmin(finite)), float(np.nanmax(finite))
    if upper <= lower:
        return np.zeros(depth.shape, dtype=np.float32)

    normalized = (depth - lower) / (upper - lower)
    return np.clip(normalized, 0.0, 1.0).astype(np.float32)


def colorize_depth(depth: np.ndarray, cmap_name: str = DEPTH_CMAP) -> np.ndarray:
    normalized = normalize_depth_for_display(depth)
    cmap = plt.get_cmap(cmap_name)
    return (cmap(normalized)[..., :3] * 255).astype(np.uint8)


def blend_overlay(frame_rgb: np.ndarray, depth_color_rgb: np.ndarray, alpha: float = OVERLAY_ALPHA) -> np.ndarray:
    blended = frame_rgb.astype(np.float32) * (1.0 - alpha) + depth_color_rgb.astype(np.float32) * alpha
    return np.clip(blended, 0, 255).astype(np.uint8)


def estimate_depth_for_frame(depth_estimator: Any, frame_rgb: np.ndarray) -> dict[str, Any]:
    image = Image.fromarray(frame_rgb)
    depth_result = depth_estimator(image)
    depth = depth_result_to_array(depth_result, image.size)
    depth_color = colorize_depth(depth)
    overlay = blend_overlay(frame_rgb, depth_color)
    return {
        "depth": depth,
        "depth_color": depth_color,
        "overlay": overlay,
        "raw_result_keys": sorted(depth_result.keys()),
    }


## 4. 表示


In [ ]:
# ------------------------------------------------------------
# セル概要: 結果表示と任意保存
# ------------------------------------------------------------
# - 1件につき、元フレーム / 深度ヒートマップ / 重ね合わせ画像を横並びで表示します。
# - SAVE_RESULT_IMAGES=True の場合は同じ比較画像を outputs 配下にも保存します。
# ------------------------------------------------------------

def format_target_title(result: dict[str, Any]) -> str:
    meta = result["metadata"]
    return (
        f"{meta['relative_path']} | "
        f"t={meta['requested_time_sec']:.3f}s | "
        f"frame={meta['frame_index']}"
    )


def result_output_path(result: dict[str, Any], output_dir: Path = OUTPUT_DIR) -> Path:
    meta = result["metadata"]
    safe_video = safe_path_part(Path(meta["relative_path"]).with_suffix("").as_posix())
    safe_time = safe_path_part(f"{meta['requested_time_sec']:.3f}s")
    return output_dir / f"{safe_video}_{safe_time}_depth.png"


def display_depth_results(results: list[dict[str, Any]]) -> None:
    if not results:
        print("No depth results to display.")
        return

    if SAVE_RESULT_IMAGES:
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    for result in results:
        fig, axes = plt.subplots(1, 3, figsize=(16, 5), constrained_layout=True)
        fig.suptitle(format_target_title(result), fontsize=12)

        axes[0].imshow(result["frame_rgb"])
        axes[0].set_title("frame")
        axes[0].axis("off")

        axes[1].imshow(result["depth_color"])
        axes[1].set_title("relative depth")
        axes[1].axis("off")

        axes[2].imshow(result["overlay"])
        axes[2].set_title("overlay")
        axes[2].axis("off")

        if SAVE_RESULT_IMAGES:
            output_path = result_output_path(result)
            fig.savefig(output_path, dpi=160)
            print(f"saved: {output_path}")

        plt.show()
        plt.close(fig)


## 5. 車体領域推定


In [ ]:
# ------------------------------------------------------------
# セル概要: 全フレーム共通の手前領域を車体領域として抽出
# ------------------------------------------------------------
# - 各深度マップの正規化値からmulti-Otsu法で遠/中/近を分け、中+近を手前領域とします。
# - 全サンプルフレームで手前に出た共通領域を車体領域候補として抽出します。
# - 連結成分ごとの bbox / 面積 / カバレッジをログ出力します。
# ------------------------------------------------------------

def otsu_threshold_from_normalized_depth(normalized: np.ndarray) -> float:
    finite = normalized[np.isfinite(normalized)]
    if finite.size == 0:
        return 1.0
    depth_u8 = (np.clip(normalized, 0.0, 1.0) * 255).astype(np.uint8)
    threshold_u8, _ = cv2.threshold(depth_u8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return float(threshold_u8 / 255.0)


def multi_otsu_thresholds_from_normalized_depth(normalized: np.ndarray) -> tuple[float, float]:
    finite = normalized[np.isfinite(normalized)]
    if finite.size == 0:
        return 1.0, 1.0

    depth_u8 = (np.clip(finite, 0.0, 1.0) * 255).astype(np.uint8)
    hist = np.bincount(depth_u8, minlength=256).astype(np.float64)
    total = hist.sum()
    if total <= 0:
        return 1.0, 1.0

    prob = hist / total
    bins = np.arange(256, dtype=np.float64)
    omega = np.cumsum(prob)
    mu = np.cumsum(prob * bins)
    mu_total = mu[-1]

    best_score = -np.inf
    best_thresholds = (0, 255)
    eps = 1e-12
    for t1 in range(1, 254):
        w0 = omega[t1]
        if w0 <= eps:
            continue
        m0 = mu[t1] / w0
        for t2 in range(t1 + 1, 255):
            w1 = omega[t2] - omega[t1]
            w2 = 1.0 - omega[t2]
            if w1 <= eps or w2 <= eps:
                continue
            m1 = (mu[t2] - mu[t1]) / w1
            m2 = (mu_total - mu[t2]) / w2
            score = w0 * (m0 - mu_total) ** 2 + w1 * (m1 - mu_total) ** 2 + w2 * (m2 - mu_total) ** 2
            if score > best_score:
                best_score = score
                best_thresholds = (t1, t2)
    return float(best_thresholds[0] / 255.0), float(best_thresholds[1] / 255.0)


def near_threshold_from_depth(depth: np.ndarray) -> float:
    normalized = normalize_depth_for_display(depth)
    method = str(DEPTH_NEAR_THRESHOLD_METHOD).strip().lower()
    if method == "multi_otsu":
        lower_threshold, _upper_threshold = multi_otsu_thresholds_from_normalized_depth(normalized)
        return lower_threshold
    if method == "otsu":
        return otsu_threshold_from_normalized_depth(normalized)
    if method == "manual":
        if DEPTH_NEAR_THRESHOLD is None:
            raise ValueError("DEPTH_NEAR_THRESHOLD must be set when DEPTH_NEAR_THRESHOLD_METHOD='manual'")
        return float(DEPTH_NEAR_THRESHOLD)
    raise ValueError("DEPTH_NEAR_THRESHOLD_METHOD must be 'multi_otsu', 'otsu', or 'manual'")


def near_mask_from_depth(depth: np.ndarray) -> tuple[np.ndarray, float]:
    normalized = normalize_depth_for_display(depth)
    threshold = near_threshold_from_depth(depth)
    return normalized >= threshold, threshold


def morphology_kernel(kernel_size: int) -> np.ndarray | None:
    kernel_size = int(kernel_size)
    if kernel_size <= 1:
        return None
    if kernel_size % 2 == 0:
        kernel_size += 1
    return cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))


def clean_vehicle_mask(mask: np.ndarray) -> np.ndarray:
    cleaned = mask.astype(np.uint8)
    open_kernel = morphology_kernel(VEHICLE_MORPH_OPEN_KERNEL)
    if open_kernel is not None:
        cleaned = cv2.morphologyEx(cleaned, cv2.MORPH_OPEN, open_kernel)
    close_kernel = morphology_kernel(VEHICLE_MORPH_CLOSE_KERNEL)
    if close_kernel is not None:
        cleaned = cv2.morphologyEx(cleaned, cv2.MORPH_CLOSE, close_kernel)
    return cleaned.astype(bool)


def connected_vehicle_regions(mask: np.ndarray) -> tuple[np.ndarray, pd.DataFrame]:
    h, w = mask.shape[:2]
    min_area_px = max(1, int(round(float(VEHICLE_MIN_COMPONENT_AREA_RATIO) * h * w)))
    labels_count, labels, stats, centroids = cv2.connectedComponentsWithStats(mask.astype(np.uint8), connectivity=8)

    kept_mask = np.zeros(mask.shape, dtype=bool)
    rows: list[dict[str, Any]] = []
    for label in range(1, labels_count):
        x = int(stats[label, cv2.CC_STAT_LEFT])
        y = int(stats[label, cv2.CC_STAT_TOP])
        width = int(stats[label, cv2.CC_STAT_WIDTH])
        height = int(stats[label, cv2.CC_STAT_HEIGHT])
        area = int(stats[label, cv2.CC_STAT_AREA])
        if area < min_area_px:
            continue
        kept_mask |= labels == label
        cx, cy = centroids[label]
        rows.append({
            "region_id": len(rows) + 1,
            "bbox_x": x,
            "bbox_y": y,
            "bbox_width": width,
            "bbox_height": height,
            "bbox_x2": x + width - 1,
            "bbox_y2": y + height - 1,
            "centroid_x": float(cx),
            "centroid_y": float(cy),
            "area_px": area,
            "coverage": float(area / max(h * w, 1)),
            "min_component_area_px": min_area_px,
        })

    regions_df = pd.DataFrame(rows)
    if len(regions_df):
        regions_df = regions_df.sort_values("area_px", ascending=False).reset_index(drop=True)
        regions_df["region_id"] = np.arange(1, len(regions_df) + 1)
    return kept_mask, regions_df


def build_vehicle_region_result(video_request: dict[str, Any], depth_results: list[dict[str, Any]]) -> dict[str, Any]:
    if not depth_results:
        raise ValueError("depth_results is empty")

    near_payloads = [near_mask_from_depth(result["depth"]) for result in depth_results]
    near_masks = [payload[0] for payload in near_payloads]
    near_thresholds = [float(payload[1]) for payload in near_payloads]
    for result, threshold in zip(depth_results, near_thresholds):
        result["near_threshold"] = threshold
    near_count = np.sum(np.stack(near_masks, axis=0), axis=0).astype(np.uint16)
    required_count = int(np.ceil(len(near_masks) * float(VEHICLE_COMMON_MIN_RATIO)))
    required_count = min(max(required_count, 1), len(near_masks))
    common_mask = near_count >= required_count
    cleaned_mask = clean_vehicle_mask(common_mask)
    vehicle_mask, regions_df = connected_vehicle_regions(cleaned_mask)

    metadata = dict(video_request["metadata"])
    sample_id = str(video_request["spec"].get("sample_id", Path(metadata["relative_path"]).stem))
    view = str(video_request["spec"].get("view", "unknown"))
    if len(regions_df):
        regions_df.insert(0, "sample_id", sample_id)
        regions_df.insert(1, "view", view)
        regions_df.insert(2, "relative_path", metadata["relative_path"])

    return {
        "video_request": video_request,
        "depth_results": depth_results,
        "near_count": near_count,
        "common_mask": common_mask,
        "vehicle_mask": vehicle_mask,
        "regions_df": regions_df,
        "required_count": required_count,
        "near_thresholds": near_thresholds,
        "sample_id": sample_id,
        "view": view,
    }


def mask_overlay(frame_rgb: np.ndarray, mask: np.ndarray, color: tuple[int, int, int] = (255, 64, 64), alpha: float = 0.45) -> np.ndarray:
    out = frame_rgb.copy().astype(np.float32)
    color_arr = np.asarray(color, dtype=np.float32)
    out[mask] = out[mask] * (1.0 - alpha) + color_arr * alpha
    return np.clip(out, 0, 255).astype(np.uint8)


def log_vehicle_region_result(result: dict[str, Any]) -> None:
    metadata = result["video_request"]["metadata"]
    depth_results = result["depth_results"]
    regions_df = result["regions_df"]
    vehicle_mask = result["vehicle_mask"]
    common_mask = result["common_mask"]
    sampled_times = [r["metadata"]["requested_time_sec"] for r in depth_results]
    near_thresholds = result.get("near_thresholds", [])
    print(f"[vehicle-region] {metadata['relative_path']}")
    print(f"  sampled_frames={len(depth_results)} required_count={result['required_count']} near_threshold_method={DEPTH_NEAR_THRESHOLD_METHOD}")
    print(f"  sampled_times_sec={', '.join(f'{time:.2f}' for time in sampled_times)}")
    if near_thresholds:
        print(f"  near_thresholds={', '.join(f'{threshold:.3f}' for threshold in near_thresholds)}")
    print(f"  common_near_area_px={int(common_mask.sum())} vehicle_area_px={int(vehicle_mask.sum())} coverage={vehicle_mask.mean():.4f}")
    if regions_df.empty:
        print("  regions: none")
        return
    for _, row in regions_df.iterrows():
        print(
            f"  region {int(row['region_id'])}: "
            f"bbox=({int(row['bbox_x'])},{int(row['bbox_y'])},{int(row['bbox_width'])},{int(row['bbox_height'])}) "
            f"area_px={int(row['area_px'])} coverage={float(row['coverage']):.4f}"
        )


def display_vehicle_region_result(result: dict[str, Any]) -> None:
    depth_results = result["depth_results"]
    first_frame = depth_results[0]["frame_rgb"]
    near_count = result["near_count"]
    vehicle_mask = result["vehicle_mask"]
    overlay = mask_overlay(first_frame, vehicle_mask)

    fig, axes = plt.subplots(1, 4, figsize=(20, 5), constrained_layout=True)
    fig.suptitle(f"vehicle region: {result['sample_id']} / {result['view']}", fontsize=12)
    axes[0].imshow(first_frame)
    axes[0].set_title("sample frame")
    axes[1].imshow(near_count, cmap="viridis", vmin=0, vmax=max(len(depth_results), 1))
    axes[1].set_title("near count")
    axes[2].imshow(vehicle_mask, cmap="gray", vmin=0, vmax=1)
    axes[2].set_title("vehicle mask")
    axes[3].imshow(overlay)
    axes[3].set_title("mask overlay")
    for ax in axes:
        ax.axis("off")
    plt.show()
    plt.close(fig)


## 6. 一括実行


In [ ]:
# ------------------------------------------------------------
# セル概要: 動画単位の等間隔深度推定と車体領域ログ出力
# ------------------------------------------------------------
# - VIDEO_TARGETS を順番に処理し、各動画から等間隔にフレームを抽出します。
# - 各フレームで深度推定を行い、全フレーム共通の手前領域を車体領域としてログ出力します。
# ------------------------------------------------------------

video_requests = iter_video_requests(VIDEO_TARGETS)
total_sampled_frames = sum(len(request["sampled_times"]) for request in video_requests)
print(f"video requests: {len(video_requests)}")
print(f"sampled frames: {total_sampled_frames}")

depth_estimator = load_depth_estimator()
all_depth_results: list[dict[str, Any]] = []
vehicle_roi_results: list[dict[str, Any]] = []

for video_index, video_request in enumerate(video_requests, start=1):
    video_path = video_request["video_path"]
    sampled_times = video_request["sampled_times"]
    print(f"[{video_index}/{len(video_requests)}] {video_path.name}: sampled_frames={len(sampled_times)}")

    video_depth_results: list[dict[str, Any]] = []
    for frame_index, time_sec in enumerate(sampled_times, start=1):
        print(f"  [{frame_index}/{len(sampled_times)}] t={time_sec:.3f}s")
        frame_rgb, metadata = read_frame_at_time(video_path, time_sec)
        depth_payload = estimate_depth_for_frame(depth_estimator, frame_rgb)
        result = {
            "frame_rgb": frame_rgb,
            "metadata": metadata,
            **depth_payload,
        }
        video_depth_results.append(result)
        all_depth_results.append(result)

    vehicle_result = build_vehicle_region_result(video_request, video_depth_results)
    log_vehicle_region_result(vehicle_result)
    vehicle_roi_results.append(vehicle_result)

frame_summary_df = pd.DataFrame([
    {
        "relative_path": result["metadata"]["relative_path"],
        "requested_time_sec": result["metadata"]["requested_time_sec"],
        "frame_index": result["metadata"]["frame_index"],
        "fps": result["metadata"]["fps"],
        "duration_sec": result["metadata"]["duration_sec"],
        "width": result["metadata"]["width"],
        "height": result["metadata"]["height"],
        "near_threshold": float(result.get("near_threshold", np.nan)),
        "depth_min": float(np.nanmin(result["depth"])),
        "depth_max": float(np.nanmax(result["depth"])),
    }
    for result in all_depth_results
])
vehicle_summary_df = pd.DataFrame([
    {
        "sample_id": result["sample_id"],
        "view": result["view"],
        "relative_path": result["video_request"]["metadata"]["relative_path"],
        "sampled_frames": len(result["depth_results"]),
        "required_count": result["required_count"],
        "near_threshold_method": DEPTH_NEAR_THRESHOLD_METHOD,
        "near_threshold_min": float(np.min(result["near_thresholds"])) if result["near_thresholds"] else np.nan,
        "near_threshold_max": float(np.max(result["near_thresholds"])) if result["near_thresholds"] else np.nan,
        "near_threshold_mean": float(np.mean(result["near_thresholds"])) if result["near_thresholds"] else np.nan,
        "common_near_area_px": int(result["common_mask"].sum()),
        "vehicle_area_px": int(result["vehicle_mask"].sum()),
        "vehicle_coverage": float(result["vehicle_mask"].mean()),
        "region_count": int(len(result["regions_df"])),
    }
    for result in vehicle_roi_results
])
vehicle_region_log_df = pd.concat([
    result["regions_df"] for result in vehicle_roi_results if len(result["regions_df"])
], ignore_index=True) if any(len(result["regions_df"]) for result in vehicle_roi_results) else pd.DataFrame()

display(frame_summary_df)
display(vehicle_summary_df)
if len(vehicle_region_log_df):
    display(vehicle_region_log_df)

if DISPLAY_PER_FRAME_DEPTH_RESULTS:
    display_depth_results(all_depth_results)
if DISPLAY_VEHICLE_REGION_RESULTS:
    for vehicle_result in vehicle_roi_results:
        display_vehicle_region_result(vehicle_result)
